# FSA and WHO Nutrition Scoring Examples

This notebook shows how the standalone `nutrition_estimator` package calculates food-item and meal-level healthiness scores.

## Example Item

We use one restaurant-style item with per-serving nutrition fields.

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while repo_root.name != "nutrition-estimator" and repo_root.parent != repo_root:
    repo_root = repo_root.parent
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from nutrition_estimator import estimate_food, estimate_meal

omelet_bites = {
    "name": "Dunkin Omelet Bites",  # optional, used only for readable output
    "nutrition": {                  # required
        "calories": 180,            # required for WHO
        "protein_g": 13,            # required for WHO
        "carbs_g": 7,               # required for WHO
        "sugar_g": 2,               # required for FSA and WHO
        "sodium_mg": 460,           # required for FSA and WHO; salt_g can replace it for FSA
        "fat_g": 11,                # required for FSA and WHO
        "saturated_fat_g": 5,       # required for FSA and WHO
        "fiber_g": 1,               # required for WHO
    },
}

omelet_bites

{'name': 'Dunkin Omelet Bites',
 'nutrition': {'calories': 180,
               'protein_g': 13,
               'carbs_g': 7,
               'sugar_g': 2,
               'sodium_mg': 460,
               'fat_g': 11,
               'saturated_fat_g': 5,
               'fiber_g': 1}}

## FSA Calculation

FSA uses sugar, sodium/salt, fat, and saturated fat. Each component receives a traffic-light score: green = 1, amber = 2, red = 3. The total range is 4-12, and lower is healthier.

In [2]:
estimate_food(omelet_bites, estimation_method="fsa")

{'item': 'Dunkin Omelet Bites',
 'estimation_method': 'fsa',
 'estimated_value': 7,
 'range': '4-12',
 'direction': 'lower_is_healthier',
 'status': 'estimated'}

## Verbose FSA Output

The default output is compact. Use `verbose=True` when you want component-level scores and normalized nutrient inputs.


In [3]:
estimate_food(omelet_bites, estimation_method="fsa", verbose=True)

{'item': 'Dunkin Omelet Bites',
 'estimation_method': 'fsa',
 'estimated_value': 7,
 'range': '4-12',
 'direction': 'lower_is_healthier',
 'status': 'estimated',
 'total': 7,
 'method': 'FSA traffic-light score from sugar, sodium/salt, fat, and '
           'saturated fat',
 'basis': 'provided',
 'missing_fields': [],
 'components': {'sugar': 1, 'salt': 2, 'fat': 2, 'saturated_fat': 2},
 'normalized_inputs': {'sugar_g': 2.0,
                       'salt_g': 1.15,
                       'fat_g': 11.0,
                       'saturated_fat_g': 5.0}}

## WHO Calculation

WHO-style scoring checks seven nutrient conditions. The range is 0-7, and higher is healthier.

In [4]:
estimate_food(omelet_bites, estimation_method="who")

{'item': 'Dunkin Omelet Bites',
 'estimation_method': 'who',
 'estimated_value': 4,
 'range': '0-7',
 'direction': 'higher_is_healthier',
 'status': 'estimated'}

## Meal-Level Aggregation

Meal-level scoring follows the MealRec+ idea: calculate each item/course score first, then average item scores.

In [5]:
snackin_bacon = {
    "name": "Dunkin Snackin Bacon",  # optional
    "nutrition": {                   # required
        "calories": 272,             # required for WHO
        "protein_g": 7,              # required for WHO
        "carbs_g": 10,               # required for WHO
        "sugar_g": 10.4,             # required for FSA and WHO
        "sodium_mg": 378,            # required for FSA and WHO; salt_g can replace it for FSA
        "fat_g": 23,                 # required for FSA and WHO
        "saturated_fat_g": 8,        # required for FSA and WHO
        "fiber_g": 0.1,              # required for WHO
    },
}

iced_coffee = {
    "name": "Dunkin Iced Coffee",  # optional
    "nutrition": {                 # required
        "calories": 158,           # required for WHO
        "protein_g": 1,            # required for WHO
        "carbs_g": 29,             # required for WHO
        "sugar_g": 33,             # required for FSA and WHO
        "sodium_mg": 98,           # required for FSA and WHO; salt_g can replace it for FSA
        "fat_g": 3,                # required for FSA and WHO
        "saturated_fat_g": 2,      # required for FSA and WHO
        "fiber_g": 0,              # required for WHO
    },
}

estimate_meal([omelet_bites, snackin_bacon, iced_coffee], estimation_method="all")

{'estimation_method': 'all',
 'items': [{'item': 'Dunkin Omelet Bites',
            'estimation_method': 'all',
            'status': 'estimated',
            'scores': {'fsa': {'estimation_method': 'fsa',
                               'estimated_value': 7,
                               'range': '4-12',
                               'direction': 'lower_is_healthier',
                               'status': 'estimated'},
                       'who': {'estimation_method': 'who',
                               'estimated_value': 4,
                               'range': '0-7',
                               'direction': 'higher_is_healthier',
                               'status': 'estimated'}}},
           {'item': 'Dunkin Snackin Bacon',
            'estimation_method': 'all',
            'status': 'estimated',
            'scores': {'fsa': {'estimation_method': 'fsa',
                               'estimated_value': 10,
                               'range': '4-12',
         

## Missing Data / No-Guess Behavior

By default, the estimator does not guess missing nutrition values. If required fields are missing, the score is `None` and the status is `insufficient_data`.


In [6]:
incomplete_item = {
    # name is optional and intentionally omitted here
    "nutrition": {   # required, but incomplete in this example
        "sugar_g": 3  # one required FSA field is present; others are missing
    }
}

estimate_food(incomplete_item, estimation_method="fsa")

{'item': None,
 'estimation_method': 'fsa',
 'estimated_value': None,
 'range': '4-12',
 'direction': 'lower_is_healthier',
 'status': 'insufficient_data',
 'missing_fields': ['fat_g', 'saturated_fat_g', 'sodium_mg_or_salt_g']}